In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_21_polymer_simulations.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Atomic Scale Simulations · Lesson 21: Polymer Physics, Coarse-Grained Models & Advanced Polymer Monte Carlo

_Session 785074c9 · Estimated time: ~115 minutes · 38 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Quantify the extreme spatial and temporal scale bridge in macromolecular dynamics—from femtosecond covalent bond vibrations to Rouse ($N^{2.2}$) and reptation ($N^{3.4}$) coil relaxation—and explain why brute-force atomistic MD is computationally intractable.
- Formulate coarse-grained polymer Hamiltonians on lattices (Self-Avoiding Walks) and in the continuum (Kremer–Grest bead-spring model with WCA excluded volume and FENE logarithmic bonds that prevent chain crossing).
- Derive and apply Flory scaling laws $\langle R^2 \rangle \propto N^{2\nu}$ across Good Solvent ($\nu \approx 0.588$, Flory $\nu = 3/(d+2)$), $\Theta$-Solvent ($\nu = 1/2$), and Poor Solvent ($\nu = 1/3$ collapsed globule) regimes.
- Analyze specialized polymer Monte Carlo move sets—Reptation ("slithering snake"), Pivot global symmetry rotations, and local kink-jump/crankshaft flips—including their correlation-time exponents and the "double cul-de-sac" non-ergodicity trap.
- Calculate Rosenbluth look-ahead weights $W_n = \prod_{i=1}^n (q_i/q)$ to overcome exponential attrition in chain growth and enforce exact detailed balance via Configurational Bias Monte Carlo (CBMC) acceptance $A = \min(1, W_{\mathrm{new}}/W_{\mathrm{old}})$.


In [2]:
ws.render_section('sec_timescale_and_coarse_graining')

## 1. The Macromolecular Timescale Chasm & Coarse-Grained Polymer Hamiltonians

Simulating long-chain polymers (`N` covalently bonded monomers) exposes one of the sharpest **multi-scale bottlenecks** in computational materials science:

1. **The Atomistic MD Timescale Chasm:**
 - Local covalent bond stretching and angle oscillations occur on a period of $\tau_{\text{vib}} \sim 10^{-13}\text{ s}$, forcing a Molecular Dynamics (MD) integration time step of $\Delta t \approx 10^{-14}\text{ s}$ ($10\text{ fs}$).
 - Yet macroscopic conformational equilibration, coil untangling, and viscosity relaxation in polymers take **seconds to hours** of real laboratory time—requiring $10^{14}$ to $10^{18}$ sequential MD force evaluations!
 - **Lecture Benchmark Estimate:** Even for a modest system of $100$ chains of $50$ monomers each ($\sim 20{,}000$ atoms including solvent) running at $1\text{ MD step/s}$, simulating just $10^{-4}\text{ s}$ ($0.1\text{ ms}$) of real time requires $10^{10}\text{ steps} \approx 10^{10}\text{ s} \approx \mathbf{300\text{ years}}$ of continuous CPU time!
 - Simultaneously, spatial scales span from monomeric bond lengths ($\sim 1\,\text{Å}$) to whole-coil pervaded volumes of $(100\,\text{Å})^3$ or larger, where solvent-mediated **hydrodynamic interactions** dominate dilute-solution transport.

2. **Coarse-Graining Strategy:**
 To bridge 14+ orders of magnitude in time, we integrate out fast atomic degrees of freedom and replace chemical repeat units with coarse-grained **blobs** or **beads**:
 - **Lattice Self-Avoiding Walk (SAW):** A polymer is represented as a connected walk of $N$ steps on a regular lattice (coordination number $q = 2d$, e.g., $q=4$ on a 2D square lattice or $q=6$ on a 3D simple cubic lattice) subject to strict **single-site occupancy** (excluded volume: no two monomers may occupy the same lattice site). Attractive nearest-neighbor non-bonded contacts $-\epsilon$ can be added to model poor solvents.
 - **Continuum Bead–Spring (Kremer–Grest) Model:** Each bead represents a Kuhn segment ("blob" of several chemical monomers). All bead pairs $(i < j)$ interact via a non-bonded excluded-volume potential (typically a truncated-and-shifted purely repulsive Weeks–Chandler–Andersen / Lennard-Jones core $\phi_{\text{LJ}}(r_{ij})$), while adjacent beads $(i, i+1)$ along the backbone are linked by a bonding potential:
 $$V(\{\mathbf{r}_i\}) = \sum_{i < j} \phi_{\text{LJ}}(r_{ij}) + \sum_{i=1}^{N-1} U_{\text{bond}}(|\mathbf{r}_i - \mathbf{r}_{i+1}|)$$

3. **Why Harmonic Springs Fail & Why FENE Bonds Are Essential:**
 A naive harmonic spring $\gamma(r) = \frac{1}{2}\kappa (r - r_0)^2$ allows large thermal bond extensions at finite $k_B T$. If two chains collide, a harmonic bond can stretch wide enough for one chain backbone to **pass directly through another chain** ("phantom chain crossing"), destroying topological entanglement and ring-polymer knotting!
 Instead, continuum polymer simulations use the **FENE (Finitely Extensible Nonlinear Elastic)** logarithmic bond potential (annotated on Lecture as a modified harmonic potential with $k = 20$ and $\sigma = 1.94\,\sigma_{\text{HS}}$, or more generally maximum extension $R_0 \approx 1.5\,\sigma$):
 $$U_{\text{FENE}}(r) = -\frac{1}{2} k R_0^2 \ln\!\left[1 - \left(\frac{r}{R_0}\right)^2\right] \quad (r < R_0)$$
 - **Small-extension limit ($r \ll R_0$):** Taylor expanding $-\ln(1 - x) = x + \frac{x^2}{2} + \frac{x^3}{3} + \cdots$ with $x = (r/R_0)^2$ yields:
 $$U_{\text{FENE}}(r) = \frac{1}{2}k r^2 + \frac{1}{4}\frac{k}{R_0^2} r^4 + \mathcal{O}(r^6)$$
 which reproduces a harmonic spring of stiffness $k$ near $r \to 0$.
 - **Finite extensibility wall ($r \to R_0^-$):** As $r \to R_0$, the argument $1 - (r/R_0)^2 \to 0^+$ and $U_{\text{FENE}}(r) \to +\infty$, erecting an **impenetrable logarithmic barrier** that strictly prevents any bond from stretching beyond $R_0$ and prevents chains from crossing through one another.


In [3]:
ws.render_section('concept_fene_bonds')

### Interactive Explorer: Kremer–Grest FENE Bond Potential vs. Harmonic Springs

Adjust the spring constant $k$, maximum bond extensibility $R_0$, and toggle the Weeks–Chandler–Andersen (WCA) hard-core repulsion to inspect how the logarithmic FENE barrier prevents phantom chain crossing while stabilizing a well-defined equilibrium bond length $r_{\mathrm{eq}} \approx 0.96\,\sigma$.

**Key Takeaway:** Even a stiff harmonic bond ($\frac{1}{2}k R_0^2$) has a finite energy at $r = R_0$, whereas $U_{\mathrm{FENE}}(r) = -\frac{1}{2}k R_0^2 \ln[1 - (r/R_0)^2]$ diverges to $+\infty$ at $r = R_0$, rigorously preserving chain connectivity and entanglement topology.

In [4]:
ws.render_section('q01_md_timescale_bottleneck')

**Q1 · easy** · 💡

In an all-atom Molecular Dynamics (MD) simulation of a polymer melt or solution, what physical motion dictates the maximum stable integration time step $\Delta t \approx 10^{-14}\text{ s}$, and how many total MD steps are required to reach macroscopic coil relaxation times of $1\text{ s}$ to $1\text{ hour}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Time estimate for MD': compare the period of local covalent vibrations ($10^{-13}\text{ s}$) against $1\text{ s} - 3600\text{ s}$ divided by $\Delta t = 10^{-14}\text{ s}$.

</div>
</details>

- [ ] **(A)** Fast local covalent bond oscillations ($\tau_{\text{vib}} \sim 10^{-13}\text{ s}$) fix $\Delta t \approx 10^{-14}\text{ s}$, requiring $10^{14}$ to $10^{18}$ sequential MD steps to reach seconds-to-hours polymer relaxation.
- [ ] **(B)** Whole-chain center-of-mass diffusion fixes $\Delta t \approx 10^{-9}\text{ s}$, requiring $10^6$ to $10^9$ steps.
- [ ] **(C)** Electronic orbital transitions fix $\Delta t \approx 10^{-18}\text{ s}$, requiring $10^{24}$ steps.
- [ ] **(D)** Rotational isomerization of side groups fixes $\Delta t \approx 10^{-6}\text{ s}$, requiring $10^6$ to $10^{10}$ steps.

In [5]:
ws.render_section('q02_md_cpu_years_estimate')

**Q2 · easy** · 💡

Consider the lecture benchmark system of $100$ polymer chains of $50$ monomers each ($\sim 20{,}000$ particles including solvent) integrated with time step $\Delta t = 10^{-14}\text{ s}$ at a computational throughput of $1\text{ step/s}$. Approximately how many **years** of wall-clock CPU time are required to simulate just $10^{-4}\text{ s}$ ($0.1\text{ ms}$) of physical real time? (Use the lecture conversion $10^{10}\text{ s} \approx 300\text{ years}$ and enter the integer number of years.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the total number of integration steps required to reach the target physical time and convert the resulting wall-clock time into years.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q03_coarse_grained_bead_meaning')

**Q3 · easy** · 💡

In a coarse-grained bead-spring polymer model (such as the Kremer–Grest model), what does a single simulated "bead" physically represent?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Bead spring model': does a bead correspond to a single carbon atom or a larger collective sub-unit?

</div>
</details>

- [ ] **(A)** A coarse-grained 'blob' (Kuhn segment) encompassing a section of several chemical monomers along the polymer chain.
- [ ] **(B)** A single valence electron shared along the conjugated polymer backbone.
- [ ] **(C)** An entire entangled polymer chain collapsed into a single point particle.
- [ ] **(D)** A solvent cage molecule that permanently binds to one specific carbon atom.

In [7]:
ws.render_section('q04_fene_taylor_expansion')

**Q4 · medium** · 💡

Consider the FENE bond potential $U_{\text{FENE}}(r) = -\frac{1}{2}k R_0^2 \ln\!\left[1 - (r/R_0)^2\right]$ for $r < R_0$. Using the Maclaurin expansion $-\ln(1 - x) = x + \frac{1}{2}x^2 + \frac{1}{3}x^3 + \cdots$, what are the leading two terms of $U_{\text{FENE}}(r)$ at small bond extensions ($r \ll R_0$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $x = (r/R_0)^2$ into $-\frac{1}{2}k R_0^2 [-\ln(1-x)] = \frac{1}{2}k R_0^2 (x + \frac{1}{2}x^2 + \cdots)$.

</div>
</details>

- [ ] **(A)** $U_{\text{FENE}}(r) = \frac{1}{2}k r^2 + \frac{k}{4 R_0^2} r^4 + \mathcal{O}(r^6)$
- [ ] **(B)** $U_{\text{FENE}}(r) = k R_0 r - \frac{1}{2}k r^2 + \mathcal{O}(r^3)$
- [ ] **(C)** $U_{\text{FENE}}(r) = \frac{1}{2}k R_0^2 r^2 + \frac{1}{2}k R_0^4 r^4 + \mathcal{O}(r^6)$
- [ ] **(D)** $U_{\text{FENE}}(r) = -\frac{1}{2}k r^2 + \frac{k}{2 R_0^2} r^4 + \mathcal{O}(r^6)$

In [8]:
ws.render_section('q05_fene_energy_numerical')

**Q5 · medium** · 💡

Using the lecture FENE bond potential $U_{\text{FENE}}(r) = -\frac{1}{2}k R_0^2 \ln\!\left[1 - (r/R_0)^2\right]$ with spring constant $k = 20$ and maximum extensibility $R_0 = 2.0$ (in reduced units), calculate $U_{\text{FENE}}(r)$ when the bond is stretched to $r = 1.6$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the dimensionless extension ratio $r/R_0$ and compute the logarithmic energy potential using the given spring constant.

</div>
</details>

_Numeric answer required._

In [9]:
ws.render_section('q06_fene_prevents_chain_crossing')

**Q6 · medium** · 💡

Why is the logarithmic divergence of $U_{\text{FENE}}(r \to R_0) \to +\infty$ (combined with a hard-core WCA excluded-volume repulsion of diameter $\sigma$ and $R_0 \approx 1.5\,\sigma$) critical for simulating entangled polymer melts and ring polymers?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about what happens to two colliding chains if a bond can stretch past $\sqrt{2}\sigma$ or $2\sigma$: can one chain pass through the bond of another?

</div>
</details>

- [ ] **(A)** It strictly bounds the maximum bond length below $R_0$, making it energetically impossible for two chains to stretch wide enough to pass through one another ('phantom chain crossing') and thereby preserving topological entanglement.
- [ ] **(B)** It eliminates all attractive and repulsive forces between non-bonded monomers so the melt behaves as an ideal gas.
- [ ] **(C)** It forces all bond angles to remain fixed at exactly $180^\circ$, converting every coil into a rigid rod.
- [ ] **(D)** It allows the MD time step $\Delta t$ to be increased to $1\text{ s}$ without numerical instability.

In [10]:
ws.render_section('q07_lattice_saw_definition')

**Q7 · easy** · 💡

Which of the following statements accurately describe the **Lattice Self-Avoiding Walk (SAW)** model of a polymer? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check and 7: how are connectivity, excluded volume, the partition function, and solvent attraction represented on a lattice?

</div>
</details>

- [ ] **(A)** Sequential monomers $i$ and $i+1$ occupy nearest-neighbor sites on a regular lattice, enforcing fixed bond connectivity.
- [ ] **(B)** No lattice site may be visited more than once by the walk, capturing hard-core excluded-volume repulsion.
- [ ] **(C)** For an athermal SAW (excluded volume only), the canonical partition function $Z_N$ is simply the total number of distinct valid $N$-step self-avoiding walks.
- [ ] **(D)** An attractive energy $-\epsilon$ between non-bonded nearest-neighbor occupied sites can be added to model poor-solvent coil-to-globule collapse.

In [11]:
ws.render_section('sec_flory_scaling_and_phases')

## 2. Polymer Phase Diagrams, Universal Flory Scaling & Rouse Relaxation

The macroscopic size of an $N$-monomer polymer coil is characterized by its **mean-square end-to-end distance** $\langle R_{ee}^2 \rangle = \langle (\mathbf{r}_N - \mathbf{r}_0)^2 \rangle$ or its **mean-square radius of gyration**:
$$\langle R_g^2 \rangle = \frac{1}{N}\sum_{i=1}^N \left\langle (\mathbf{r}_i - \mathbf{r}_{\text{cm}})^2 \right\rangle = \frac{1}{2N^2}\sum_{i=1}^N \sum_{j=1}^N \left\langle (\mathbf{r}_i - \mathbf{r}_j)^2 \right\rangle$$
In the asymptotic long-chain limit ($N \gg 1$), both length scales obey the universal **Flory scaling law**:
$$\langle R_{ee}^2 \rangle \propto N^{2\nu} \quad \Longleftrightarrow \quad R_g \equiv \sqrt{\langle R_g^2 \rangle} \propto a\,N^\nu$$
where $\nu$ is the universal **Flory size exponent**, which depends only on spatial dimensionality $d$ and the balance between monomer–monomer excluded volume and solvent-mediated attraction:

| Polymer Phase / Solvent Regime | Physical Balance | Exact / Numerical $\nu$ ($d=3$) | Flory Mean-Field $\nu$ | $\langle R^2 \rangle$ Scaling ($d=3$) |
|:--- |:--- |:---: |:---: |:---: |
| **Good Solvent (Swollen SAW)** | Repulsive excluded volume dominates; coil swells | $\nu \approx \mathbf{0.588}$ | $\nu_{\text{F}} = \frac{3}{d+2} = \frac{3}{5} = \mathbf{0.60}$ | $\langle R^2 \rangle \propto N^{1.176}$ |
| **$\Theta$-Solvent (Theta Point)** | Attraction exactly cancels two-body repulsion (ideal RW) | $\nu = \mathbf{0.500}$ ($1/2$) | $\nu = 1/2$ | $\langle R^2 \rangle = N a^2$ |
| **Poor Solvent (Collapsed Globule)** | Monomer attraction dominates; dense liquid-like drop | $\nu = \frac{1}{d} = \mathbf{0.333}$ ($1/3$) | $\nu = 1/3$ | $\langle R^2 \rangle \propto N^{2/3}$ |
| **Rigid Rod / 1D Walk** | Fully stretched chain or $d=1$ SAW | $\nu = \mathbf{1.000}$ | $\nu = 1$ | $\langle R^2 \rangle \propto N^2$ |

### Flory's Mean-Field Free Energy Derivation of $\nu = \frac{3}{d+2}$
Flory estimated the equilibrium coil size $R$ in a good solvent by balancing two competing free-energy contributions in $d$ dimensions:
1. **Entropic Elastic Stretching Penalty:** Stretching an ideal Gaussian chain ($\langle R_0^2 \rangle = N a^2$) to radius $R$ reduces conformational entropy:
 $$\frac{F_{\text{el}}(R)}{k_B T} \sim \frac{R^2}{N a^2}$$
2. **Excluded-Volume Two-Body Repulsion:** Inside the coil volume $V \sim R^d$, the mean monomer density is $\rho \sim N / R^d$, so the repulsive two-body interaction energy scales as $V v_{\text{excl}} \rho^2$:
 $$\frac{F_{\text{rep}}(R)}{k_B T} \sim v_{\text{excl}} R^d \left(\frac{N}{R^d}\right)^2 = v_{\text{excl}} \frac{N^2}{R^d}$$
Minimizing $F_{\text{tot}}(R) = F_{\text{el}}(R) + F_{\text{rep}}(R)$ with respect to $R$ ($\partial F_{\text{tot}}/\partial R = 0$) gives:
$$\frac{R}{N a^2} \sim \frac{N^2}{R^{d+1}} \quad \Longrightarrow \quad R^{d+2} \propto N^3 \quad \Longrightarrow \quad \boxed{R \propto N^{\frac{3}{d+2}} \implies \nu_{\text{Flory}} = \frac{3}{d+2} \quad (1 \le d \le 4)}$$
- For **$d = 1$**: $\nu_{\text{Flory}} = 3/3 = 1$ (exact; a 1D self-avoiding walk cannot turn back!).
- For **$d = 2$**: $\nu_{\text{Flory}} = 3/4 = 0.75$ (exact in 2D!).
- For **$d = 3$**: $\nu_{\text{Flory}} = 3/5 = 0.60$, remarkably close to the renormalization-group / Monte Carlo value $\nu \approx 0.588$.
- For **$d \ge 4$ (upper critical dimension $d_c = 4$)**: two random walks rarely intersect, so excluded volume becomes irrelevant and $\nu = 1/2$.

### Dynamical Scaling & Rouse / Reptation Relaxation Times
Why does coil swelling make Molecular Dynamics and local Monte Carlo so slow?
- In an unentangled **Rouse chain**, a monomer diffuses with local mobility $\sim 1/N$ over a distance $R \sim N^\nu$, requiring a global conformational **relaxation time**:
 $$\tau_{\text{Rouse}} \sim \frac{R^2}{D_{\text{cm}}} \sim \frac{N^{2\nu}}{N^{-1}} = N^{2\nu + 1}$$
 For a 3D SAW ($\nu \approx 0.588 \approx 0.6$), this gives the lecture scaling **$\tau_{\text{relax}} \sim N^{2.2}$** (or $N^2$ for an ideal $\Theta$-chain).
- In an **entangled polymer melt** ($N > N_e$, de Gennes tube model), chains are confined to topological tubes of contour length $L_{\text{tube}} \propto N$ with curvilinear tube diffusivity $D_{\text{tube}} \propto N^{-1}$, giving $\tau_{\text{rep}} \sim L_{\text{tube}}^2 / D_{\text{tube}} \propto N^3$ (and experimentally $\tau_{\text{relax}} \propto N^{3.4}$ due to contour-length fluctuations).
- Beyond linear chains, polymer physics explores diverse **macromolecular topologies**, including **ring polymers** (no free ends!) and **star/branched polymers** ($f$ arms radiating from a central core).


In [12]:
ws.render_section('concept_flory_scaling')

### Interactive Explorer: Flory Size Scaling & Coil Conformations Across Solvent Regimes

Switch between **Poor Solvent** ($\nu = 1/d$), **$\Theta$-Solvent** ($\nu = 0.5$), **Good Solvent** ($\nu \approx 0.588$ in 3D, $0.75$ in 2D), and **Rigid Rod** ($\nu = 1.0$) while varying the chain length $N$ and dimension $d$ to compare the log-log scaling of $R_g(N)$, coil conformations, and Rouse relaxation times $\tau_{\mathrm{relax}} \sim N^{2\nu+1}$.

**Key Takeaway:** Cooling a polymer solution from a good solvent through the $\Theta$-point into a poor solvent triggers a continuous **coil-to-globule phase transition** in which $R_g \sim N^{3/5} \to N^{1/2} \to N^{1/3}$ in $d=3$.

In [13]:
ws.render_section('q08_flory_exponent_good_solvent_3d')

**Q8 · easy** · 💡

For a linear polymer chain in a **3D good solvent** (where repulsive excluded-volume interactions dominate), how does the mean-square end-to-end distance $\langle (\mathbf{r}_N - \mathbf{r}_0)^2 \rangle \propto N^{2\nu}$ scale with chain length $N$, and what is the numerical value of the Flory exponent $\nu$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Polymer Phases': compare the SAW exponent $\nu$ against the ideal random walk exponent $0.5$.

</div>
</details>

- [ ] **(A)** $\nu \approx 0.588$ (Flory mean-field approximation $\nu = 3/5 = 0.6$), so the coil swells larger than an ideal random walk ($\langle R^2 \rangle \propto N^{1.176}$).
- [ ] **(B)** $\nu = 0.500$ exactly, because excluded volume cancels out in three dimensions.
- [ ] **(C)** $\nu = 1/3 \approx 0.333$, because repulsive interactions compress the polymer into a dense sphere.
- [ ] **(D)** $\nu = 2.2$, matching the Rouse relaxation time exponent.

In [14]:
ws.render_section('q09_flory_mean_field_derivation_2d')

**Q9 · medium** · 💡

Flory's mean-field balance between entropic elastic stretching $F_{\text{el}}/k_B T \sim R^2 / N$ and two-body excluded-volume repulsion $F_{\text{rep}}/k_B T \sim N^2 / R^d$ predicts $\nu_{\text{Flory}} = \frac{3}{d+2}$ for $d \le 4$. What is the value of $\nu_{\text{Flory}}$ for a polymer adsorbed flat onto a **2D surface ($d = 2$)**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate Flory's dimensional scaling prediction for the coil expansion exponent in two dimensions.

</div>
</details>

_Numeric answer required._

In [15]:
ws.render_section('q10_theta_solvent_compensation')

**Q10 · medium** · 💡

What happens to the polymer conformation and size scaling right at the **$\Theta$-point (Theta temperature)** as attractive monomer–monomer interactions are tuned against hard-core excluded volume?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

'Right at collapse point—walks are uncorrelated random walks.'

</div>
</details>

- [ ] **(A)** Effective two-body monomer attraction cancels excluded-volume repulsion (second virial coefficient $B_2 = 0$), so the chain obeys quasi-ideal random-walk scaling $R_g \propto N^{1/2}$ ($\nu = 0.5$).
- [ ] **(B)** The polymer crystallizes into a 1D rigid rod with $R_g \propto N^1$ ($\nu = 1$).
- [ ] **(C)** All covalent bonds break simultaneously, dissociating the chain into free monomers.
- [ ] **(D)** The radius of gyration becomes completely independent of $N$ ($\nu = 0$).

In [16]:
ws.render_section('q11_poor_solvent_globule_exponent')

**Q11 · medium** · 💡

When a polymer in 3D is cooled well below the $\Theta$-temperature into a **poor solvent**, monomer–monomer attraction dominates and the chain collapses into a compact globule. Why does its radius of gyration scale as $R_g \propto N^{1/3}$ ($\nu = 1/3$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In a compact liquid-like drop of uniform internal monomer density $\rho_0$, how does the volume $V \sim R_g^3$ relate to the total number of monomers $N$?

</div>
</details>

- [ ] **(A)** A collapsed globule packs monomers at a uniform liquid-like interior density $\rho \sim N / R_g^3 = \text{const}$, which immediately requires $R_g^3 \propto N \implies R_g \propto N^{1/3}$.
- [ ] **(B)** Only one-third of the monomers remain bonded while the other two-thirds evaporate.
- [ ] **(C)** The chain projects onto a single spatial axis out of the three Cartesian coordinates.
- [ ] **(D)** Three-body repulsive forces scale as $R_g^3$ with zero entropic elasticity.

In [17]:
ws.render_section('q12_scaling_ratio_calculation')

**Q12 · medium** · 💡

Suppose a polymer chain in a 3D good solvent ($\nu = 0.588$) has its degree of polymerization increased by a factor of $8$ (from $N$ to $8N$). By what factor does its root-mean-square end-to-end distance $\sqrt{\langle R_{ee}^2 \rangle} \propto N^\nu$ increase?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Raise the degree-of-polymerization expansion factor to the good-solvent scaling exponent $\nu$.

</div>
</details>

_Numeric answer required._

In [18]:
ws.render_section('q13_md_rouse_relaxation_scaling')

**Q13 · medium** · 💡

unentangled MD / local-move polymer relaxation time scales as $\tau_{\text{relax}} \propto N^{2.2}$ in a 3D good solvent ($\tau \sim N^{2\nu + 1}$ with $\nu \approx 0.6$). If you increase the chain length by a factor of $10$ (from $N = 50$ to $N = 500$), by what factor does the conformational relaxation time $\tau_{\text{relax}}$ increase?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Scale the relaxation time by the chain-length expansion ratio raised to the dynamic scaling exponent.

</div>
</details>

_Numeric answer required._

In [19]:
ws.render_section('q14_polymer_topologies')

**Q14 · easy** · 💡

Which of the following polymer architectures—highlighted alongside linear chains and star polymers—possesses **zero free chain ends**, making standard end-slithering reptation moves impossible without modifying the move set?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect the three sketches at the bottom of: linear, rings, stars.

</div>
</details>

- [ ] **(A)** Ring (cyclic) polymers, where monomer $N$ is covalently bonded back to monomer $1$ to form a closed loop.
- [ ] **(B)** 3-arm star polymers, which have three free chain ends radiating from one central core.
- [ ] **(C)** Linear homopolymers in a $\Theta$-solvent.
- [ ] **(D)** Short linear oligomers of length $N = 10$.

In [20]:
ws.render_section('sec_reptation_pivot_and_ergodicity')

## 3. Polymer Monte Carlo Move Menus: Reptation, Pivot & Ergodicity Traps

If we only care about **equilibrium statistical mechanics** (coil dimensions, structure factors, phase coexistence, osmotic equation of state) rather than exact Newtonian trajectories, we can abandon Molecular Dynamics and design **unphysical Monte Carlo (MC) trial moves** that leap across conformational space orders of magnitude faster while rigorously obeying detailed balance!

### 1. Local Moves (End-Bond Rotation, Kink-Jump / Corner Flip, Crankshaft)
- **End-bond move:** Rotate the first or last bond to a new neighbor site.
- **Kink-jump (corner flip):** If monomer $i$ forms a $90^\circ$ corner between $i-1$ and $i+1$, flip monomer $i$ across the diagonal to the opposite corner of the elementary square.
- **Crankshaft move:** Rotate a 2-monomer U-shaped segment by $90^\circ$ or $180^\circ$ around the axis connecting its flanking monomers.
- **Limitation:** Because local moves only displace $1$–$2$ monomers at a time, they mimic physical Rouse dynamics and suffer the same slow global decorrelation time $\tau_{\text{local}} \sim \mathcal{O}(N^{2\nu+1}) \approx \mathcal{O}(N^{2.2})$ (or $\mathcal{O}(N^3)$ in melts).

### 2. Reptation ("Slithering Snake" Algorithm — Wall & Mandel, 1975)
Inspired by de Gennes' reptation theory of entangled melts:
- **Trial Move:** Pick one of the two chain ends at random (head or tail), **delete** the terminal monomer at that end, and **append** a new monomer onto the *other* end in a randomly chosen bond direction $\theta$ (or lattice neighbor).
- **Detailed Balance Subtlety:** To preserve time-reversal symmetry (detailed balance), you must either choose which end acts as the growing head with equal $50/50$ probability at every step, or implement a symmetric "bounce" rule (staying in the old state and recording it again in the Markov chain whenever the proposed site is occupied).
- **Computational Efficiency:** Every accepted step shifts all monomer indices along the existing chain contour in $\mathcal{O}(1)$ operations, sampling the bonding interaction directly while the Metropolis acceptance probability depends *only* on the change in non-bonded potential at the deleted/added ends!
- **Scaling Advantage:** Because the chain slithers along its own contour of length $N$ in random forward/backward steps of $\pm 1$ monomer, it vacates its initial tube after $\sim N^2$ elementary steps, reducing the conformational decorrelation time to **$\tau_{\text{reptation}} \sim \mathcal{O}(N^2)$**—and crucially, **it still works in dense multi-chain melts!**
- **The Non-Ergodicity Trap ("Double Cul-de-Sac"):** Is pure lattice reptation ergodic? **No!** As noted "*), a single chain (or multi-chain system) can slither into a **"double cul-de-sac" (double dead-end)** configuration where *both* chain ends are completely surrounded by already-occupied interior monomers of the chain ($q_{\text{open}} = 0$ at both ends). Once in a double cul-de-sac, no valid reptation move exists—nor can pure reptation ever enter such a valid SAW configuration from outside—violating ergodicity!

### 3. The Pivot Algorithm (Lal 1969; Madras & Sokal 1988)
For a **single isolated polymer chain in dilute solution**:
- **Trial Move:** Pick a monomer $k \in \{1, \dots, N-1\}$ uniformly at random as a **pivot point**, and apply a random global symmetry transformation (rotation by $\pm 90^\circ, 180^\circ$, reflection, or continuous 3D angle $\theta$) to the entire subchain $(k+1, \dots, N)$ relative to monomer $k$.
- **Accept or Reject:** Check whether the rotated subchain overlaps with the unrotated subchain $(1, \dots, k)$ (or changes non-bonded energy $\Delta U$); accept with the standard Metropolis probability $\min(1, e^{-\beta \Delta U})$.
- **Spectacular Efficiency for Global Observables:** Because a single accepted pivot move reorients $\mathcal{O}(N)$ monomers simultaneously, the end-to-end vector decorrelates in **$\tau_{\text{pivot}} \sim \mathcal{O}(N^{0.2})$** accepted moves (measured in macroscopic sweeps)—making the pivot algorithm the **most efficient method known for single-chain global equilibrium properties**!
- **Limitations of Pure Pivot:**
 1. In **dense polymer melts** or compact poor-solvent globules, rotating half a chain almost always collides with surrounding chains, driving the acceptance probability to zero ($A \to 0$).
 2. Because pivot moves apply rigid-body rotations on a lattice, they never alter bond lengths or local 1-step bond vectors if not combined with local moves.

### 4. The Hybrid Move Menu Rule
To guarantee **ergodicity** (reaching every valid conformation in chain space) while relaxing *both* short-wavelength local bond modes and long-wavelength global coil vectors, practical polymer MC simulations always employ a **mixture of moves** (e.g., combining Pivot + Reptation + Crankshaft/Kink-jump).


In [21]:
ws.render_section('concept_reptation_pivot')

### Interactive Explorer: Reptation, Pivot Rotations & The Double Cul-de-Sac Trap

Select a polymer Monte Carlo move mode (**Reptation**, **Pivot Rotation**, **Crankshaft/Kink-Jump**, or **Double Cul-de-Sac Trap**) and vary the pivot monomer index $k$ and rotation angle $\theta$ to observe how each move transforms the 2D square-lattice chain and when self-avoidance collisions or non-ergodic traps occur.

**Key Takeaway:** Reptation achieves $\mathcal{O}(N^2)$ tube renewal in melts via $\mathcal{O}(1)$ end updates, whereas the Pivot algorithm achieves $\mathcal{O}(N^{0.2})$ global decorrelation for dilute single chains. Mixing these with local crankshaft moves eliminates the double cul-de-sac ergodicity trap.

In [22]:
ws.render_section('q15_reptation_move_mechanism')

**Q15 · easy** · 💡

How does an elementary trial step of the **Reptation ("slithering snake") Monte Carlo** algorithm update an $N$-monomer polymer chain?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

'Cut off one end and stick onto the other end.'

</div>
</details>

- [ ] **(A)** It deletes one terminal monomer from a randomly chosen chain end and appends a new monomer in a random bond direction onto the opposite chain end.
- [ ] **(B)** It translates all $N$ monomers rigidly by the same random Cartesian vector $\Delta \mathbf{r}$.
- [ ] **(C)** It cuts the polymer in half at monomer $N/2$ and swaps the two halves.
- [ ] **(D)** It simultaneously perturbs the $(x,y,z)$ coordinates of all $N$ monomers using Gaussian white noise.

In [23]:
ws.render_section('q16_reptation_decorrelation_scaling')

**Q16 · medium** · 💡

Why does the conformational decorrelation time (in elementary MC steps) of the **reptation algorithm** scale as $\mathcal{O}(N^2)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each elementary slithering step shifts the chain contour by $\pm 1$ monomer along a tube of length $N$ monomers. How many random $\pm 1$ steps are required to diffuse a distance of $N$ monomers along the 1D contour?

</div>
</details>

- [ ] **(A)** Because choosing either end at random causes the chain to perform a 1D random walk of step size $\pm 1$ monomer along its contour of length $N$, requiring $\sim N^2$ steps to diffuse completely out of its initial tube.
- [ ] **(B)** Because evaluating the distance between the two chain ends requires an $N \times N$ matrix multiplication at every step.
- [ ] **(C)** Because only $1/N^2$ of proposed end moves avoid overlapping with the immediately adjacent monomer.
- [ ] **(D)** Because the radius of gyration scales as $R_g \propto N^2$ in a good solvent.

In [24]:
ws.render_section('q17_reptation_double_cul_de_sac')

**Q17 · medium** · 💡

Why is the pure **lattice reptation algorithm** strictly **non-ergodic** for self-avoiding walks?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'But Ergodic? Not always (what if both ends get trapped?)' and inspect the 'Double Cul-de-Sac Trap' in the interactive explorer above.

</div>
</details>

- [ ] **(A)** Because there exist valid self-avoiding configurations ('double cul-de-sac' traps) in which all nearest-neighbor sites of both chain ends are occupied by interior monomers, preventing the chain from ever entering or leaving those states via end-slithering alone.
- [ ] **(B)** Because reptation violates conservation of total monomer number $N$.
- [ ] **(C)** Because reptation can only move the chain forward and never backward, violating microscopic reversibility.
- [ ] **(D)** Because reptation always collapses a good-solvent chain into a $\nu = 1/3$ globule.

In [25]:
ws.render_section('q18_reptation_bounce_detailed_balance')

**Q18 · hard** · 💡

the lecture notes state: *"Choose end at random or 'bounce' with rejection."* Why does a naive reptation scheme that **always** grows at end $N$ (moving strictly forward along the chain) violate detailed balanceunless ends are chosen at random (or reversed upon rejection)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the detailed balance condition $\pi(o) T(o \to n) = \pi(n) T(n \to o)$: if state $n$ is obtained from state $o$ by deleting monomer $1$ and adding monomer $N+1$, how can the Markov chain return directly from $n$ to $o$ in one step?

</div>
</details>

- [ ] **(A)** Because moving from state $o$ to state $n$ by deleting the tail and growing at the head can only be reversed in a single step ($n \to o$) if the algorithm is also allowed to delete the newly grown head monomer and regrow at the old tail!
- [ ] **(B)** Because always growing at the head causes the chain length $N$ to grow to infinity.
- [ ] **(C)** Because the Boltzmann factor $e^{-\beta E}$ is different for the head monomer than for the tail monomer.
- [ ] **(D)** Because forward-only growth changes the lattice coordination number $q$.

In [26]:
ws.render_section('q19_pivot_algorithm_relaxation_exponent')

**Q19 · easy** · 💡

how does the end-to-end distance relaxation time of the **Pivot algorithm** scale with chain length $N$ for an isolated self-avoiding polymer chain?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look directly at the fourth bullet point: 'Most efficient method for a single chain. Exponent of relaxation of end-end distance is...'

</div>
</details>

- [ ] **(A)** $\tau_{\text{relax}} \propto N^{0.2}$, making it the most efficient Monte Carlo method for sampling global conformations of a single dilute chain.
- [ ] **(B)** $\tau_{\text{relax}} \propto N^{2.2}$, identical to Molecular Dynamics.
- [ ] **(C)** $\tau_{\text{relax}} \propto N^{3.4}$, matching entangled tube reptation.
- [ ] **(D)** $\tau_{\text{relax}} \propto e^{0.2 N}$, growing exponentially with chain length.

In [27]:
ws.render_section('q20_pivot_melt_failure')

**Q20 · medium** · 💡

Although the Pivot algorithm is extraordinarily fast for an isolated polymer coil in a good solvent, why does it perform poorly in a **dense polymer melt** or a tightly **collapsed poor-solvent globule**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What happens when you rigidly swing a subchain of hundreds of monomers through a densely packed box filled with other monomers?

</div>
</details>

- [ ] **(A)** Rigidly rotating a macroscopic subchain of $\sim N/2$ monomers through a dense melt or compact globule almost always causes hard-core steric overlaps with neighboring monomers, driving the acceptance probability exponentially to zero.
- [ ] **(B)** Pivot moves do not conserve the total number of monomers $N$ in a multi-chain box.
- [ ] **(C)** In a dense melt, the Flory exponent becomes $\nu = 0$, so rotation angles are undefined.
- [ ] **(D)** Pivot moves require calculating hydrodynamic Oseen tensors at every step.

In [28]:
ws.render_section('q21_hybrid_move_menu_rationale')

**Q21 · medium** · 💡

Why do state-of-the-art polymer Monte Carlo simulations typically employ a **hybrid mixture of moves** (combining local crankshaft/kink-jump flips, reptation, and pivot moves) rather than relying on a single move type? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall 'Ergodic questions arise: Can you go everywhere in chain space? Make a mixture of moves.'

</div>
</details>

- [ ] **(A)** Combining local interior flips (crankshaft/kink-jump) with reptation or pivot moves eliminates non-ergodic traps such as double cul-de-sacs, ensuring the Markov chain can reach all valid conformations.
- [ ] **(B)** Global pivot moves rapidly decorrelate long-wavelength observables (like $R_{ee}$ and $R_g$), while local crankshaft/kink-jump moves efficiently equilibrate short-wavelength local bond orientations.
- [ ] **(C)** Any convex combination (probabilistic mixture) of transition kernels that individually satisfy detailed balance with respect to the Boltzmann distribution also satisfies detailed balance.
- [ ] **(D)** Mixing two non-Boltzmann moves that violate detailed balance always cancels out their bias without needing Metropolis acceptance tests.

In [29]:
ws.render_section('sec_rosenbluth_and_cbmc')

### 1. The Attrition Problem in Simple ("Blind") Polymer Growth
Suppose we try to generate an $N$-step Self-Avoiding Walk (SAW) from scratch on a lattice of coordination number $q$ by starting at the origin and blindly picking one of the $q$ neighbor directions (or $q-1$ non-reversing directions) at each step, throwing away the entire walk whenever an overlap occurs:
- Because each step has a finite probability of colliding with an already visited site, the fraction of walks that survive to length $N$ decays **exponentially**:
 $$P_{\text{survive}}(N) \propto e^{-\lambda N}$$
- This is the **exponential attrition problem**: for $N \sim 100$–$1000$, virtually $100\%$ of blind walks terminate in self-intersections before reaching length $N$.

### 2. The Rosenbluth–Rosenbluth (1955) Biased Growth Method
To avoid stepping onto occupied sites, Rosenbluth and Rosenbluth introduced **look-ahead importance sampling**:
1. At growth step $i$ (adding monomer $i$ onto monomer $i-1$), scan all $q$ nearest-neighbor lattice sites around monomer $i-1$ and count the number of **unoccupied ("open") moves** $q_i \in \{0, 1, \dots, q\}$.
2. If $q_i = 0$, the walk has hit a dead-end trap; set weight $W_n = 0$ and terminate.
3. If $q_i > 0$, choose one of the $q_i$ unoccupied neighbor sites uniformly at random with selection probability $p_{\text{select}}(i) = \frac{1}{q_i}$ (instead of the unbiased blind probability $\frac{1}{q}$).
4. **Correcting the Bias with the Rosenbluth Weight $W_n$:**
 Because we forced the walk into open sites with probability $\prod_{i=1}^n (1/q_i)$ instead of the uniform prior $(1/q)^n$, conformations running through crowded regions (small $q_i$) are **over-sampled** relative to open conformations (large $q_i$). To obtain unbiased ensemble averages $\langle A \rangle$ over all SAWs, each generated $n$-step chain $j$ must be weighted by its **Rosenbluth weight**:
 $$W_n = W_{n-1} \left[\frac{q_i}{q}\right] = \prod_{i=1}^n \left(\frac{q_i}{q}\right), \qquad \langle R^2 \rangle = \frac{\sum_j W^{(j)}_n R_j^2}{\sum_j W^{(j)}_n}$$
 *(Note: Some conventions define $w_i = q_i$ without the constant $q^{-n}$ denominator since $q^n$ cancels in any ratio $\sum_j W_j A_j / \sum_j W_j$ or $W_{\text{new}}/W_{\text{old}}$.)*
5. **High-Variance Limitation for Very Long Chains & Branching (PERM):**
 For very long chains ($N \gg 100$), the product $W_n = \prod_{i=1}^n (q_i/q)$ is a product of many random factors, so its distribution becomes extremely broad (log-normal): a tiny handful of high-weight chains dominate the average $\sum W_j R_j^2 / \sum W_j$. As noted, algorithms like **PERM (Pruned-Enriched Rosenbluth Method)** clone/branch high-weight walks (halving each clone's weight) and prune low-weight walks to keep weight fluctuations bounded!

### 3. Configurational Bias Monte Carlo (CBMC — Siepmann & Frenkel, 1992)
Instead of growing full chains independently from scratch, **Configurational Bias Monte Carlo (CBMC)** embeds Rosenbluth look-ahead growth *inside* a Markov Chain Monte Carlo update (; Frenkel & Smit Chapters 11 & 13):
1. **Select a Chain Segment to Regrow:** Pick a polymer chain and cut off a terminal segment of $m$ monomers (or an interior segment).
2. **Grow the New Segment ($\text{new}$):** Regrow the $m$ monomers step-by-step using Rosenbluth look-ahead sampling, recording its new Rosenbluth weight:
 $$W_{\text{new}} = \prod_{i=1}^m \left(\frac{q_i^{(\text{new})}}{q}\right) \quad \left(\text{or in continuum/thermal systems: } W_{\text{new}} = \prod_{i=1}^m \left[\frac{1}{k}\sum_{j=1}^k e^{-\beta u_i(\mathbf{b}_j)}\right]\right)$$
 where at each step $i$, $k$ trial bond directions $\{\mathbf{b}_1, \dots, \mathbf{b}_k\}$ are proposed and direction $\mathbf{b}_j$ is selected with probability $P_i(\mathbf{b}_j) = \frac{e^{-\beta u_i(\mathbf{b}_j)}}{\sum_{\ell=1}^k e^{-\beta u_i(\mathbf{b}_\ell)}}$.
3. **Retrace the Old Segment ($\text{old}$):** Compute the Rosenbluth weight $W_{\text{old}} = \prod_{i=1}^m \left(q_i^{(\text{old})} / q\right)$ that the *existing* old segment would have had (generating $k-1$ phantom trial directions plus the 1 actual old bond at each step $i$ in the thermal/continuum case).
4. **Super-Detailed Balance Acceptance Rule:**
 Because the probability of proposing the new path is $\mathcal{P}_{\text{gen}}(\text{new}) = \prod_{i=1}^m \frac{1}{q_i^{(\text{new})}} = \frac{1}{q^m W_{\text{new}}}$ while its equilibrium weight is constant (or $\propto \prod e^{-\beta u_i}$), detailed balance $\pi(\text{old})\mathcal{P}_{\text{gen}}(\text{new}) A(\text{old}\to\text{new}) = \pi(\text{new})\mathcal{P}_{\text{gen}}(\text{old}) A(\text{new}\to\text{old})$ simplifies miraculously to:
 $$\boxed{\text{acc. Prob} = A(\text{old} \to \text{new}) = \min\!\left[1,\, \frac{W_{\text{new}}}{W_{\text{old}}}\right]}$$
 Every factor of the look-ahead selection probability cancels against the Boltzmann factor, yielding an **exact, ergodic Markov chain** that threads long polymer segments into dense melts and porous media with high acceptance!


In [30]:
ws.render_section('concept_cbmc_rosenbluth')

### Interactive Explorer: Rosenbluth Weights & Configurational Bias MC (CBMC) Acceptance

Explore how Rosenbluth look-ahead growth accumulates step weights $W_n = \prod_{i=1}^n (q_i/q)$ compared to blind growth attrition, and adjust the weight ratio $W_{\mathrm{new}}/W_{\mathrm{old}}$ to see how CBMC computes its exact Metropolis-like acceptance probability $A = \min(1, W_{\mathrm{new}}/W_{\mathrm{old}})$.

**Key Takeaway:** Rosenbluth growth steers polymer segments away from steric overlaps by choosing among the $q_i$ open neighbor sites, and CBMC removes that look-ahead bias inside a Markov chain by accepting the regrown segment with $A(\mathrm{old}\to\mathrm{new}) = \min(1, W_{\mathrm{new}}/W_{\mathrm{old}})$.

In [31]:
ws.render_section('q22_blind_growth_attrition')

**Q22 · easy** · 💡

Why does the naive ("blind") polymer growth algorithm—which grows a chain step-by-step at random and discards the entire walk whenever a self-intersection occurs—fail for long chains ($N \gg 100$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the survival probability $P_{\text{survive}}(N)$ scales as $N$ increases when each step has a nonzero probability of overlap.

</div>
</details>

- [ ] **(A)** Because the probability that a blind walk avoids all self-intersections decays exponentially with chain length ($P_{\text{survive}} \propto e^{-\lambda N}$), a phenomenon known as exponential attrition.
- [ ] **(B)** Because blind growth biases the surviving walks toward collapsed globules with $\nu = 1/3$.
- [ ] **(C)** Because a lattice of coordination number $q$ can only hold at most $q^2$ monomers.
- [ ] **(D)** Because blind growth requires computing the full $N \times N$ Hessian matrix at every step.

In [32]:
ws.render_section('q23_rosenbluth_weight_definition')

**Q23 · easy** · 💡

In the lattice **Rosenbluth growth algorithm**, suppose the lattice coordination number is $q$, and at growth step $i$ there are $q_i$ unoccupied ("open") nearest-neighbor sites available. How is the next site chosen, and how is the cumulative **Rosenbluth weight** $W_n$ updated?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the formula: $W_n = W_{n-1} [q_i / q]$ where $q_i = \text{# of open moves}$.

</div>
</details>

- [ ] **(A)** One of the $q_i$ open sites is chosen with probability $1/q_i$, and the weight is updated via $W_n = W_{n-1} \left(\frac{q_i}{q}\right) = \prod_{i=1}^n \left(\frac{q_i}{q}\right)$.
- [ ] **(B)** One of all $q$ neighbor sites is chosen with probability $1/q$, and $W_n = W_{n-1} + q_i$.
- [ ] **(C)** The open site closest to the origin is chosen deterministically, and $W_n = (q / q_i)^n$.
- [ ] **(D)** One of the $q_i$ open sites is chosen with probability $q_i / q$, and $W_n = 1$ for all chains.

In [33]:
ws.render_section('q24_rosenbluth_weight_numerical')

**Q24 · medium** · 💡

A 3-step self-avoiding polymer segment is grown on a **2D square lattice ($q = 4$)** using the Rosenbluth algorithm with $W_0 = 1$. Suppose the number of open neighbor sites at steps $i = 1, 2, 3$ is $q_1 = 4$, $q_2 = 3$, and $q_3 = 2$. What is the resulting Rosenbluth weight $W_3 = \prod_{i=1}^3 (q_i / q)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Multiply the step survival fractions $q_i / q$ over the three growth steps.

</div>
</details>

_Numeric answer required._

In [34]:
ws.render_section('q25_rosenbluth_why_reweight')

**Q25 · medium** · 💡

Suppose two valid $N$-step self-avoiding walks, Walk A and Walk B, are generated by the Rosenbluth growth algorithm. Walk A passes through a very crowded, tightly coiled region where most steps only had $q_i = 1$ open site, whereas Walk B passes through an open region where most steps had $q_i = 3$ open sites. Before reweighting by $W_N$, which walk is **more likely to be generated** by the Rosenbluth sampling procedure, and how does $W_N$ correct this?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the path generation probability $P_{\text{gen}} = \prod_{i=1}^N (1/q_i)$ for small $q_i$ vs. Large $q_i$.

</div>
</details>

- [ ] **(A)** Walk A is generated with higher probability ($P_{\text{gen}} = \prod (1/q_i)$ is larger when $q_i$ is small), so Rosenbluth reweighting assigns Walk A a correspondingly smaller weight $W_N = \prod (q_i/q) = (q^N P_{\text{gen}})^{-1}$ so that $P_{\text{gen}} W_N = q^{-N}$ is equal for all valid SAWs.
- [ ] **(B)** Walk B is generated with higher probability, so Walk B receives a smaller Rosenbluth weight $W_N$.
- [ ] **(C)** Both walks are generated with identical probability $q^{-N}$, so $W_N$ is only needed when non-bonded attractions are nonzero.
- [ ] **(D)** Walk A has zero probability of being generated because $q_i = 1$ terminates the walk.

In [35]:
ws.render_section('q26_rosenbluth_weighted_estimator')

**Q26 · medium** · 💡

In a Rosenbluth growth simulation of $M = 3$ polymer chains, you record the following Rosenbluth weights $W_j$ and squared end-to-end distances $R_j^2$:
- Chain 1: $W_1 = 0.20$, $R_1^2 = 10.0$
- Chain 2: $W_2 = 0.50$, $R_2^2 = 20.0$
- Chain 3: $W_3 = 0.30$, $R_3^2 = 30.0$

Using the lecture estimator $\langle R^2 \rangle = \frac{\sum_j W_j R_j^2}{\sum_j W_j}$, what is the unbiased estimate of $\langle R^2 \rangle$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute the weighted average of the squared end-to-end distances normalized by the sum of weights.

</div>
</details>

_Numeric answer required._

In [36]:
ws.render_section('q27_rosenbluth_branching_perm')

**Q27 · medium** · 💡

When growing very long polymers with the standalone Rosenbluth method, the weights $W_n = \prod_{i=1}^n (q_i/q)$ fluctuate over many orders of magnitude so that a few rare trajectories dominate the sample average. What algorithmic technique is introduced to overcome this weight-fluctuation bottleneck?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the fourth bullet: 'Use... When weights fluctuate too much.'

</div>
</details>

- [ ] **(A)** Branching (cloning/enriching trajectories whose weights grow too large and pruning trajectories whose weights fall too low, as in PERM).
- [ ] **(B)** Replacing the Rosenbluth weight $W_n$ with $\ln W_n$ inside the numerator of $\langle R^2 \rangle$.
- [ ] **(C)** Freezing the first $90\\%$ of the chain into a straight rod.
- [ ] **(D)** Increasing the MD integration time step $\Delta t$.

In [37]:
ws.render_section('q28_cbmc_acceptance_formula')

**Q28 · easy** · 💡

In **Configurational Bias Monte Carlo (CBMC)**, a section of a polymer chain is regrown using Rosenbluth look-ahead sampling with weight $W_{\text{new}}$, while the old section has retraced Rosenbluth weight $W_{\text{old}}$. What is the exact CBMC acceptance probability $A(\text{old} \to \text{new})$ shown?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the boxed formula at the bottom of.

</div>
</details>

- [ ] **(A)** $A(\text{old} \to \text{new}) = \min\\!\left[1,\\, \frac{W_{\text{new}}}{W_{\text{old}}}\right]$
- [ ] **(B)** $A(\text{old} \to \text{new}) = \min\\!\left[1,\\, \frac{W_{\text{old}}}{W_{\text{new}}}\right]$
- [ ] **(C)** $A(\text{old} \to \text{new}) = \frac{W_{\text{new}}}{W_{\text{new}} + W_{\text{old}}} e^{-\beta (W_{\text{new}} - W_{\text{old}})}$
- [ ] **(D)** $A(\text{old} \to \text{new}) = 1$ always, because Rosenbluth growth never overlaps.

In [38]:
ws.render_section('q29_cbmc_acceptance_numerical')

**Q29 · easy** · 💡

During a Configurational Bias Monte Carlo (CBMC) update on a 3D cubic lattice ($q = 6$), you regrow a 3-bond end segment. The newly grown segment has open-site counts $(q_1, q_2, q_3)_{\text{new}} = (3, 4, 3)$, while retracing the old segment yields $(q_1, q_2, q_3)_{\text{old}} = (5, 4, 3)$. What is the CBMC acceptance probability $A(\text{old} \to \text{new}) = \min(1, W_{\text{new}}/W_{\text{old}})$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the ratio of the product of open neighbors along the new trial path to that along the old path.

</div>
</details>

_Numeric answer required._

In [39]:
ws.render_section('q30_cbmc_continuum_generalization')

**Q30 · hard** · 💡

notes that CBMC is *"Easily generalized to continuum models."* In a continuum bead-spring polymer at inverse temperature $\beta = 1/(k_B T)$, there are infinitely many possible bond directions rather than $q$ discrete lattice sites. How does continuum CBMC perform step $i$ of the Rosenbluth regrowth?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall Frenkel & Smit Chapters 11 & 13: how do we replace summing over all $q$ lattice neighbors when directions are continuous?

</div>
</details>

- [ ] **(A)** It generates a finite set of $k$ random trial bond directions $\\{\mathbf{b}_1, \dots, \mathbf{b}_k\\}$ (sampled from the bonded distribution), selects one direction $\mathbf{b}_j$ with Boltzmann probability $e^{-\beta u_i(\mathbf{b}_j)} / \sum_{\ell=1}^k e^{-\beta u_i(\mathbf{b}_\ell)}$, and accumulates the step Rosenbluth factor $w_i = \frac{1}{k}\sum_{\ell=1}^k e^{-\beta u_i(\mathbf{b}_\ell)}$.
- [ ] **(B)** It discretizes all of 3D space into a cubic grid of spacing $0.001\\,\text{Å}$ and sums over all grid points in the box.
- [ ] **(C)** It integrates the Newtonian equations of motion for $k$ femtoseconds and sets $W_{\text{new}} = 1$.
- [ ] **(D)** It chooses the single direction of steepest descent $-\nabla U$ deterministically with zero random trials.

In [40]:
ws.render_section('sec_synthesis_and_comparison')

## 5. Synthesis: Choosing the Right Polymer Simulation Method Across Regimes

Let us synthesize the scaling laws and algorithmic trade-offs covered across Lesson 21:

| Simulation Method | Primary Use Case | Conformational Relaxation Scaling $\tau_{\text{relax}}$ | Key Advantage | Primary Limitation |
|:--- |:--- |:---: |:--- |:--- |
| **All-Atom MD** | Local chemistry, solvation shells, $<100\text{ ns}$ kinetics | $\tau \sim N^{2.2}$ to $N^{3.4}$ ($\Delta t = 10^{-14}\text{ s}$) | Exact physical dynamics & explicit solvent | $\sim 300\text{ CPU years}$ for $10^{-4}\text{ s}$ of $20{,}000$ atoms |
| **Coarse-Grained MD (Kremer–Grest FENE + WCA)** | Entangled melt rheology, tube reptation dynamics | $\tau_{\text{Rouse}} \sim N^{2.2}$, $\tau_{\text{rep}} \sim N^{3}$–$N^{3.4}$ | Preserves real topological entanglement dynamics | Still slow for very large $N$ equilibration |
| **Local Lattice MC (Kink-jump / Crankshaft)** | Short-scale bond relaxation & Rouse dynamics | $\tau \sim \mathcal{O}(N^{2\nu+1}) \approx \mathcal{O}(N^{2.2})$ | Simple, physical Rouse-like local dynamics | Slow global coil decorrelation |
| **Reptation MC ("Slithering Snake")** | Dense polymer melts & concentrated solutions | $\tau \sim \mathcal{O}(N^2)$ elementary steps | $\mathcal{O}(1)$ update cost; works in dense melts | Unphysical dynamics; **non-ergodic** alone (double cul-de-sac) |
| **Pivot Algorithm** | Single isolated chain in dilute good/$\Theta$ solvent | $\tau \sim \mathcal{O}(N^{0.2})$ sweeps | Fastest global decorrelation known | Acceptance $\to 0$ in dense melts / globules; doesn't relax bond lengths |
| **Rosenbluth + CBMC** | Regrowing segments in dense melts, branched chains, phase equilibria | High acceptance for $m$-segment regrowth | Overcomes exponential attrition; exact detailed balance $A = \min(1, W_{\text{new}}/W_{\text{old}})$ | Weight variance grows for very large $m$ without branching |


In [41]:
ws.render_section('q31_method_selection_dilute_saw')

**Q31 · easy** · 💡

You want to compute the equilibrium Flory exponent $\nu$ and mean-square end-to-end distance $\langle R_{ee}^2 \rangle$ of a **single isolated $N = 2000$ self-avoiding polymer chain** in a good solvent to high statistical precision. Which algorithm provides the fastest decorrelation of global conformations?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the relaxation time exponents in the synthesis table: $N^{2.2}$ vs. $N^2$ vs. $N^{0.2}$.

</div>
</details>

- [ ] **(A)** The Pivot algorithm (mixed with occasional local moves), because its global end-to-end autocorrelation time scales as $\mathcal{O}(N^{0.2})$ in dilute solution.
- [ ] **(B)** All-atom Molecular Dynamics with explicit water molecules and $\Delta t = 10^{-14}\text{ s}$.
- [ ] **(C)** Pure blind random growth without Rosenbluth weights.
- [ ] **(D)** Pure local kink-jump Monte Carlo without global moves.

In [42]:
ws.render_section('q32_method_selection_dense_melt_dynamics')

**Q32 · medium** · 💡

Suppose your research goal is to measure the **physical time-dependent mean-square displacement $g_1(t) = \langle [\mathbf{r}_i(t) - \mathbf{r}_i(0)]^2 \rangle$ and viscosity** of an entangled polymer melt to test de Gennes' tube reptation theory. Why must you use **coarse-grained Molecular Dynamics (with FENE + WCA potentials)** or purely **local MC moves** rather than the Pivot or CBMC algorithms?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Do Pivot and CBMC moves follow physical continuous trajectories, or do they teleport chains across topological obstacles?

</div>
</details>

- [ ] **(A)** Because Pivot and CBMC perform non-local, unphysical conformational jumps that bypass topological entanglement barriers; measuring real physical transport and entanglement kinetics requires continuous FENE+WCA MD trajectories (or strictly local non-crossing moves).
- [ ] **(B)** Because Pivot and CBMC do not sample the canonical Boltzmann distribution at equilibrium.
- [ ] **(C)** Because FENE+WCA Molecular Dynamics has a relaxation time of $\mathcal{O}(N^{0.2})$.
- [ ] **(D)** Because CBMC can only be applied to 1D chains.

In [43]:
ws.render_section('q33_hydrodynamic_solvent_effects')

**Q33 · medium** · 💡

Note that: *"Solvent is important. Hydrodynamic effects dominate."* In a dilute polymer solution, why does omitting explicit solvent molecules (or hydrodynamic coupling) alter the physical relaxation time scaling of a polymer coil?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In a real fluid, moving one monomer creates a velocity field that drags nearby monomers along (Zimm hydrodynamics), whereas without solvent momentum transport each bead feels independent friction (Rouse model).

</div>
</details>

- [ ] **(A)** Moving a monomer in a real solvent induces long-range velocity fields (hydrodynamic interactions) that couple the motion of distant segments of the coil, whereas naive vacuum/implicit-solvent dynamics treats friction as purely local and independent.
- [ ] **(B)** Solvent molecules change the spatial dimensionality of the simulation box from $d=3$ to $d=2$.
- [ ] **(C)** Without explicit solvent molecules, the FENE potential $U_{\text{FENE}}(r)$ no longer diverges at $r = R_0$.
- [ ] **(D)** Hydrodynamic momentum conservation alters the static equilibrium Flory exponent $\nu$ of an athermal SAW.

In [44]:
ws.render_section('q34_saw_partition_function_temperature')

**Q34 · medium** · 💡

In a pure **athermal Lattice Self-Avoiding Walk (SAW)** where the potential energy is $U = 0$ for every non-overlapping walk and $U = +\infty$ for any overlapping walk, how does the equilibrium Flory exponent $\nu$ depend on temperature $T$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Write down the Boltzmann factor $e^{-U / (k_B T)}$ when $U$ can only take the values $0$ or $+\infty$.

</div>
</details>

- [ ] **(A)** It is completely independent of temperature $T$ ($\nu \approx 0.588$ at all $T > 0$), because $e^{-0/k_B T} = 1$ and $e^{-\infty/k_B T} = 0$, so every valid SAW has identical weight regardless of $T$; temperature dependence only enters when finite non-bonded attractions $-\epsilon$ are added.
- [ ] **(B)** It decreases linearly with temperature as $\nu(T) = 1 - k_B T$.
- [ ] **(C)** It jumps from $\nu = 0.588$ to $\nu = 1/3$ at $k_B T = 1$ even when $\epsilon = 0$.
- [ ] **(D)** It diverges to infinity as $T \to 0$.

In [45]:
ws.render_section('q35_radius_of_gyration_ratio_good_vs_poor')

**Q35 · medium** · 💡

Consider a long polymer chain of $N = 1000$ monomers in $d = 3$ dimensions. Using the Flory scaling relation $R_g \approx a N^\nu$ (with unit prefactor $a = 1$), calculate the ratio $\frac{R_g(\text{Good Solvent, }\nu = 0.6)}{R_g(\text{Poor Solvent, }\nu = 1/3)}$ to see how dramatically a swollen coil shrinks upon collapsing into a globule.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Take the ratio of the two scaling expressions evaluated at $N = 1000$ using their respective exponents.

</div>
</details>

_Numeric answer required._

In [46]:
ws.render_section('q36_cbmc_detailed_balance_derivation')

**Q36 · hard** · 💡

Let us verify why $A(\text{old} \to \text{new}) = \min(1, W_{\text{new}}/W_{\text{old}})$ satisfies **detailed balance** for an athermal lattice SAW where every valid configuration has equal equilibrium probability $\pi(\text{old}) = \pi(\text{new})$. If an $m$-bond segment is regrown by choosing among $q_i$ open neighbors at each step $i$, what is the exact transition proposal probability $\mathcal{P}_{\text{gen}}(\text{old} \to \text{new})$ of generating a specific new path, expressed in terms of $q$ and $W_{\text{new}} = \prod_{i=1}^m (q_i^{(\text{new})} / q)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

At each step $i = 1, \dots, m$, the specific bond along the new path is chosen with probability $1 / q_i^{(\text{new})}$. Multiply these $m$ probabilities together and substitute $W_{\text{new}} = \prod_{i=1}^m q_i^{(\text{new})} / q^m$.

</div>
</details>

- [ ] **(A)** $\mathcal{P}_{\text{gen}}(\text{old} \to \text{new}) = \prod_{i=1}^m \frac{1}{q_i^{(\text{new})}} = \frac{1}{q^m W_{\text{new}}}$, so $\frac{\mathcal{P}_{\text{gen}}(\text{new} \to \text{old})}{\mathcal{P}_{\text{gen}}(\text{old} \to \text{new})} = \frac{W_{\text{new}}}{W_{\text{old}}}$.
- [ ] **(B)** $\mathcal{P}_{\text{gen}}(\text{old} \to \text{new}) = q^m W_{\text{new}}$, so the ratio of proposal probabilities is $W_{\text{old}} / W_{\text{new}}$.
- [ ] **(C)** $\mathcal{P}_{\text{gen}}(\text{old} \to \text{new}) = q^{-m}$ regardless of $W_{\text{new}}$.
- [ ] **(D)** $\mathcal{P}_{\text{gen}}(\text{old} \to \text{new}) = 1 - W_{\text{new}}$.

In [47]:
ws.render_section('q37_crankshaft_move_geometry')

**Q37 · easy** · 💡

Inspect the local polymer move illustrated in the center of ("How to move polymers"). When a single interior right-angle corner (kink) of a 2D square-lattice polymer flips across the unit square while keeping the rest of the chain fixed, how many monomer coordinates actually change?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the black chain on the left of with the chain on the right (highlighted in orange/red): how many vertices moved to a new lattice site?

</div>
</details>

- [ ] **(A)** Only $1$ interior corner monomer moves to the diagonally opposite site of the unit square (or $2$ adjacent monomers in a 3D U-shaped crankshaft rotation), leaving all other $N-1$ (or $N-2$) monomers untouched.
- [ ] **(B)** All $N$ monomers shift by one lattice spacing along the chain contour.
- [ ] **(C)** Every monomer from the flipped corner to the chain end rotates by $90^\circ$.
- [ ] **(D)** Two new monomers are inserted into the chain backbone.

In [48]:
ws.render_section('q38_upper_critical_dimension_flory')

**Q38 · hard** · 💡

According to Flory's formula $\nu_{\text{Flory}} = \frac{3}{d+2}$, at what **upper critical spatial dimension $d_c$** does the swollen self-avoiding walk exponent $\nu_{\text{Flory}}$ drop down to match the ideal random walk exponent $\nu_{\text{RW}} = \frac{1}{2}$ (above which excluded-volume interactions become asymptotically irrelevant for long chains)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Equate Flory's dimensional formula for the swollen coil exponent to the ideal random walk value and solve for the dimension.

</div>
</details>

_Numeric answer required._

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()